In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
import pandas as pd
import numpy as np
import re
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler

In [22]:
train_data = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test_data = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')
all_data = pd.concat([train_data,test_data])

In [15]:
train_data.isnull().sum()

PassengerId           0
Survived              0
Pclass                0
Name                  0
Sex                   0
Age                   0
SibSp                 0
Parch                 0
Ticket                0
Fare                  0
Cabin               687
Embarked              0
Title                 0
Title_encoded         0
FamilySize            0
Sex_encoded           0
Prefix_ticket         0
Embarked_encoded      0
FareGroup             0
Deck                  0
Deck_encoded          0
dtype: int64

In [29]:
test_data.isnull().sum()

PassengerId           0
Pclass                0
Name                  0
Sex                   0
Age                   0
SibSp                 0
Parch                 0
Ticket                0
Fare                  0
Cabin               327
Embarked              0
Title                 0
Title_encoded         0
FamilySize            0
Sex_encoded           0
Prefix_ticket         0
Embarked_encoded      0
FareGroup             0
Deck                  0
Deck_encoded          0
dtype: int64

In [23]:
## --- Title extraction ---
def process_titles(df, name_col='Name', title_col='Title', threshold=10, rare_label='Rare'):
    df[title_col] = df[name_col].apply(lambda x: x.split(',')[1].split('.')[0].strip())
    title_counts = df[title_col].value_counts()
    rare_titles = title_counts[title_counts < threshold].index.tolist()
    df[title_col] = df[title_col].replace(rare_titles, rare_label)
    return df

train_data = process_titles(train_data, name_col='Name', threshold=10)
test_data = process_titles(test_data, name_col='Name', threshold=10)

title_mapping = {'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4}
train_data['Title_encoded'] = train_data['Title'].map(title_mapping).fillna(4)
test_data['Title_encoded'] = test_data['Title'].map(title_mapping).fillna(4)

# --- Family size ---
train_data['FamilySize'] = train_data['SibSp'] + train_data['Parch'] + 1
test_data['FamilySize'] = test_data['SibSp'] + train_data['Parch'] + 1

# --- Sex encoding ---
train_data['Sex_encoded'] = train_data['Sex'].map({'male': 0, 'female': 1})
test_data['Sex_encoded'] = test_data['Sex'].map({'male': 0, 'female': 1})

# --- Ticket prefix ---
def extract_ticket_prefix_v3(ticket):
    if pd.isna(ticket):
        return -1
    ticket = str(ticket).strip()
    if ticket.startswith('PC'):
        return 1
    elif re.match(r'^[A-Za-z]+[A-Za-z0-9/\.]*\s*\d*', ticket):
        return 4
    elif ticket[0].isdigit():
        if int(ticket[0]) > 3:
            return 3
        else:
            return int(ticket[0])
    else:
        return -1

train_data['Prefix_ticket'] = train_data['Ticket'].apply(extract_ticket_prefix_v3)
test_data['Prefix_ticket'] = test_data['Ticket'].apply(extract_ticket_prefix_v3)


In [24]:
# adding the missing values
#missing Embarked
train_data['Embarked'].fillna(train_data['Embarked'].mode()[0], inplace=True)

## --- Embarked encoding ---
train_data['Embarked_encoded'] = train_data['Embarked'].fillna('S').map({'S':0, 'C':1, 'Q':2})
test_data['Embarked_encoded'] = test_data['Embarked'].fillna('S').map({'S':0, 'C':1, 'Q':2})

#Fair for test:
mean_fare_by_class = train_data.groupby('Pclass')['Fare'].mean()
test_data['Fare'] = test_data['Fare'].fillna(test_data['Pclass'].map(mean_fare_by_class))
#Fair Group:
# --- Fare groups (quantiles) ---
train_data['FareGroup'], bins = pd.qcut(
    train_data['Fare'], q=4, labels=['1','2','3','4'],
    duplicates='drop', retbins=True
)
# --- Fare groups (quantiles) ---
test_data['FareGroup'], bins = pd.qcut(
    test_data['Fare'], q=4, labels=['1','2','3','4'],
    duplicates='drop', retbins=True
)

train_data['FareGroup'] = train_data['FareGroup'].astype(int)
test_data['FareGroup'] = test_data['FareGroup'].astype(int)

#-------------Deck----Cabin-------------
train_data['Deck'] = train_data['Cabin'].apply(lambda x: x[0] if pd.notna(x) else 'Unknown')
test_data['Deck'] = test_data['Cabin'].apply(lambda x: x[0] if pd.notna(x) else 'Unknown')

# --- Encode Deck (ordinal) ---

deck_mapping = {
    'Unknown': 0, 'A': 1, 'B': 2, 'C': 3, 'D': 4,
    'E': 5, 'F': 6, 'G': 7, 'T': 8
}
train_data['Deck_encoded'] = train_data['Deck'].map(deck_mapping).fillna(0).astype(int)
test_data['Deck_encoded'] = test_data['Deck'].map(deck_mapping).fillna(0).astype(int)

train_data['Sex_encoded'] = train_data['Sex'].map({'male': 0, 'female': 1})
test_data['Sex_encoded'] = test_data['Sex'].map({'male': 0, 'female': 1})

# --- Family size ---
train_data['FamilySize'] = train_data['SibSp'] + train_data['Parch'] + 1
test_data['FamilySize'] = test_data['SibSp'] + test_data['Parch'] + 1







/tmp/ipykernel_55/2677022096.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_data['Embarked'].fillna(train_data['Embarked'].mode()[0], inplace=True)


In [25]:
# ======================
# 3. AGE IMPUTATION WITH RANDOMFOREST (replaces median by title)
# ======================
# Define features for age prediction (must be available for all rows)
age_features = [
    'Pclass', 'Sex_encoded', 'Title_encoded', 'FamilySize',
    'Deck_encoded', 'FareGroup', 'Embarked_encoded', 'SibSp', 'Parch'
]

# Separate rows with known and missing age
known_age = train_data[train_data['Age'].notna()].copy()
missing_age = train_data[train_data['Age'].isna()].copy()

print(missing_age.info())
X_age_known = known_age[age_features]
#print(X_age_known.info())
y_age_known = known_age['Age']

X_age_missing = missing_age[age_features]

#print(X_age_missing.info())

# Train RandomForestRegressor
age_model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
age_model.fit(X_age_known, y_age_known)

predicted_ages = age_model.predict(X_age_missing)
#print(predicted_ages)
predicted_ages = np.clip(predicted_ages, 0, 100)
train_data.loc[train_data['Age'].isna(), 'Age'] = predicted_ages

#print(test_data.head())

#missing_age = test_data[test_data['Age'].isna()].copy()
#print(missing_age.head())
#X_age_missing = missing_age[age_features]
#print(X_age_missing.head())



<class 'pandas.core.frame.DataFrame'>
Index: 177 entries, 5 to 888
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   PassengerId       177 non-null    int64  
 1   Survived          177 non-null    int64  
 2   Pclass            177 non-null    int64  
 3   Name              177 non-null    object 
 4   Sex               177 non-null    object 
 5   Age               0 non-null      float64
 6   SibSp             177 non-null    int64  
 7   Parch             177 non-null    int64  
 8   Ticket            177 non-null    object 
 9   Fare              177 non-null    float64
 10  Cabin             19 non-null     object 
 11  Embarked          177 non-null    object 
 12  Title             177 non-null    object 
 13  Title_encoded     177 non-null    int64  
 14  FamilySize        177 non-null    int64  
 15  Sex_encoded       177 non-null    int64  
 16  Prefix_ticket     177 non-null    int64  
 17  Em

In [28]:
#test_data
missing_age = test_data[test_data['Age'].isna()].copy()
X_age_missing = missing_age[age_features]
predicted_ages = age_model.predict(X_age_missing)
predicted_ages = np.clip(predicted_ages, 0, 100)
test_data.loc[test_data['Age'].isna(), 'Age'] = predicted_ages


In [36]:
# ======================
#  ADDITIONAL FEATURES (optional)
# ======================
train_data['IsChild'] = (train_data['Age'] < 16).astype(int)
train_data['IsAlone'] = (train_data['FamilySize'] == 1).astype(int)
test_data['IsChild'] = (test_data['Age'] < 16).astype(int)
test_data['IsAlone'] = (test_data['FamilySize'] == 1).astype(int)

In [37]:
# ======================
# PREPARE FEATURE SET FOR SURVIVAL MODEL
# ======================
features = [
    'Pclass', 'Age', 'Fare', 'Deck_encoded', 'FareGroup',
    'Title_encoded', 'Prefix_ticket', 'Sex_encoded', 'Embarked_encoded',
    'FamilySize', 'IsChild', 'IsAlone'
]

X_train = train_data[features]
y_train = train_data['Survived']
X_test = test_data[features]

In [38]:
# ======================
# FEATURE SCALING
# ======================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ======================
# HYPERPARAMETER TUNING WITH CROSS‑VALIDATION
# ======================
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'class_weight': [None, 'balanced'],
    'solver': ['lbfgs']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=4, penalty='l2'),
    param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)
grid.fit(X_train_scaled, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=LogisticRegression(max_iter=1000, random_state=4),
             n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 10, 100],
                         'class_weight': [None, 'balanced'],
                         'solver': ['lbfgs']},
             scoring='accuracy')

In [47]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test_scaled)
print(y_pred)
submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Survived': y_pred
})
submission.to_csv('/kaggle/working/submission.csv', index=False)

[0 0 0 0 1 0 1 0 1 0 0 0 1 0 1 1 0 0 1 1 0 0 1 0 1 0 1 0 0 0 0 0 0 0 1 0 1
 1 0 0 0 0 0 1 1 0 0 0 1 0 1 0 1 1 0 0 0 0 0 1 0 0 0 1 1 1 1 0 0 1 1 0 1 1
 1 1 0 1 0 1 1 0 0 0 0 0 1 1 1 1 1 0 1 0 0 0 1 0 1 0 1 0 0 0 1 0 0 0 0 0 0
 1 1 1 1 0 0 1 1 1 1 0 1 0 0 1 0 1 0 0 0 1 0 0 0 0 0 0 1 0 0 1 0 0 0 0 1 0
 0 0 1 0 0 0 0 0 1 1 0 1 1 1 1 0 0 1 0 0 1 1 0 0 0 0 0 1 1 0 1 1 0 0 1 0 1
 0 1 0 0 0 0 0 0 0 1 0 1 1 0 1 1 0 1 1 0 0 1 0 1 0 0 0 0 1 0 0 1 0 1 0 1 0
 1 0 1 1 0 1 0 0 1 1 0 0 0 0 0 0 1 1 1 1 0 0 0 0 1 0 1 1 1 0 1 0 0 0 0 0 1
 0 0 0 1 1 0 0 0 0 1 0 0 0 1 1 0 1 0 0 0 0 1 0 1 1 1 0 0 0 0 0 0 1 0 0 0 0
 1 0 0 0 0 0 0 0 1 1 0 1 0 1 0 0 0 1 1 1 0 0 0 0 0 0 0 0 1 0 1 0 0 0 1 0 0
 1 0 0 0 0 0 0 0 0 0 1 0 1 0 1 0 1 1 0 0 0 1 0 1 0 0 1 0 1 1 0 1 0 0 1 1 0
 0 1 0 0 1 1 0 0 0 0 0 0 1 1 0 1 0 0 0 0 1 1 0 0 0 1 0 1 0 0 1 0 1 1 0 0 0
 0 1 1 1 1 1 0 1 0 0 1]
